# H&M Transaction Data: Product Recommendations 03
## Decision Forest Models

### Models Tested
Two decision forest models are evaluated as alternatives to the neural network:
- **Random Forest** — decision trees trained in parallel
- **Histogram Gradient Boosting** — sequential 

### Hyperparameter Tuning
Both models are tuned using `RandomizedSearchCV` optimizing for F1 score on the positive class. 

### Results
| Model | F1 (Purchase) | Precision | Recall | PR-AUC |
|---|---|---|---|---|
| Random Forest | **0.79** | 0.71 | 0.88 | 0.922 |
| Gradient Boosting | 0.78 | 0.69 | 0.89 | 0.924 |
| Neural Network | 0.74 | 0.65 | 0.86 | 0.874 |

Comparing F1 for the positive class (purchased=1) across all three models, **Random Forest achieves the best result** and is the preferred model. 
The decision forest models outperform the neural network across all metrics, likely because tree-based models are better suited to tabular data with mixed feature types, sentinel-imputed values, and binary indicator columns.
Comparing F1 for the positive class (purchased=1) across all three models, **Random Forest achieves the best result** and is the preferred model.

### Concerns
The best F1 from cross-validation (0.91) is notably higher than the validation set F1 (0.76), suggesting the model may be overfitting to the training data. The test set result (0.79) is the most reliable estimate of true generalization performance. Further regularization or a broader hyperparameter search may help close this gap.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn import ensemble
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, f1_score, precision_score, recall_score

import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

import os
import sys
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# data processing classes
from src.customer_features import CustomerFeatureEngineer
from src.product_features import ProductFeatureEngineer
from src.recommendation_training import RecommendationTrainingBuilder


## Load Processed Data

In [2]:
# Load data from pickle
data_path = Path("../data")
processed_data_path = data_path / 'processed' / 'product_recommendation'

with open(processed_data_path / 'X_train_base.pkl', 'rb') as f:
    X_train = pickle.load(f)
with open(processed_data_path / 'y_train_base.pkl', 'rb') as f:
    y_train = pickle.load(f)
with open(processed_data_path / 'X_val_base.pkl', 'rb') as f:
    X_val = pickle.load(f)
with open(processed_data_path / 'y_val_base.pkl', 'rb') as f:
    y_val = pickle.load(f)
with open(processed_data_path / 'X_test_base.pkl', 'rb') as f:
    X_test = pickle.load(f)
with open(processed_data_path / 'y_test_base.pkl', 'rb') as f:
    y_test = pickle.load(f)

print(f"{X_train.shape=}")
print(f"{y_train.shape=}")
print(f"{X_val.shape=}")
print(f"{y_val.shape=}")
print(f"{X_test.shape=}")
print(f"{y_test.shape=}")
print()
# Check what data was loaded
print(f"X_train type: {type(X_train)}")
print(f"X_train shape: {X_train.shape}")
print(f"X_train memory: {X_train.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nData types:")
print(X_train.dtypes)

X_train.shape=(238936, 47)
y_train.shape=(238936,)
X_val.shape=(230262, 47)
y_val.shape=(230262,)
X_test.shape=(221238, 47)
y_test.shape=(221238,)

X_train type: <class 'pandas.core.frame.DataFrame'>
X_train shape: (238936, 47)
X_train memory: 85.68 MB

Data types:
sales_last_7_days                        float64
garment_Blouses                            int64
days_since_last_sale                     float64
avg_days_between_purchases               float64
garment_Socks and Tights                   int64
FN                                       float64
garment_Shoes                              int64
avg_transaction_value                    float64
garment_Jersey Fancy                       int64
garment_Outdoor                            int64
garment_Trousers Denim                     int64
garment_Swimwear                           int64
days_since_last_purchase                 float64
garment_Under-, Nightwear                  int64
days_since_first_sale                    float64

## Model Building - Random Forest

In [3]:
random_state=67

def predict_and_report_val(model, name):
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1]

    print(f"======= {name} — Validation ========")
    print(f"  PR-AUC:    {average_precision_score(y_val, y_proba):.4f}")
    print(f"  F1:        {f1_score(y_val, y_pred):.4f}")
    print(f"  Precision: {precision_score(y_val, y_pred):.4f}")
    print(f"  Recall:    {recall_score(y_val, y_pred):.4f}")
    print()
    print(classification_report(y_val, y_pred, target_names=['No Purchase', 'Purchase']))

In [4]:
random_state=67

def predict_and_report_test(model, name):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(f"======= {name} — Test ========")
    print(f"  PR-AUC:    {average_precision_score(y_test, y_proba):.4f}")
    print(f"  F1:        {f1_score(y_test, y_pred):.4f}")
    print(f"  Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"  Recall:    {recall_score(y_test, y_pred):.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=['No Purchase', 'Purchase']))

In [5]:
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    'n_estimators': [550, 600, 700, 800],
    'max_depth': [28, 30, 32, 36],
    'min_samples_leaf': [8, 10, 15, 20],
}

rf_search = RandomizedSearchCV(
    ensemble.RandomForestClassifier(
        random_state=random_state,
        n_jobs=-1
    ),
    param_distributions=param_distributions,
    n_iter=10,
    scoring='f1',
    cv=3,
    random_state=random_state,
    verbose=1
)

rf_search.fit(X_train, y_train)

print(f"Best params: {rf_search.best_params_}")
print(f"Best F1: {rf_search.best_score_:.4f}")

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params: {'n_estimators': 700, 'min_samples_leaf': 10, 'max_depth': 28}
Best F1: 0.9110


In [6]:
predict_and_report_val(rf_search.best_estimator_, "Random Forest")

======= Random Forest — Validation ========
  PR-AUC:    0.9121
  F1:        0.7657
  Precision: 0.6761
  Recall:    0.8828

              precision    recall  f1-score   support

 No Purchase       0.98      0.92      0.94    191885
    Purchase       0.68      0.88      0.77     38377

    accuracy                           0.91    230262
   macro avg       0.83      0.90      0.86    230262
weighted avg       0.93      0.91      0.91    230262



In [7]:
predict_and_report_test(rf_search.best_estimator_, "Random Forest")

======= Random Forest — Test ========
  PR-AUC:    0.9213
  F1:        0.7898
  Precision: 0.7156
  Recall:    0.8813

              precision    recall  f1-score   support

 No Purchase       0.98      0.93      0.95    184365
    Purchase       0.72      0.88      0.79     36873

    accuracy                           0.92    221238
   macro avg       0.85      0.91      0.87    221238
weighted avg       0.93      0.92      0.92    221238



## Model Building - Gradient Boosting

In [8]:
param_distributions = {
    'max_iter': [150, 200, 250, 300],
    'max_depth': [20, 25, 30],
    'min_samples_leaf': [10, 20, 30],
    'learning_rate': [0.05, 0.1, 0.2]
}

hgb_search = RandomizedSearchCV(
    ensemble.HistGradientBoostingClassifier(random_state=67),
    param_distributions=param_distributions,
    scoring='f1',
    cv=3,
    random_state=random_state,
    verbose=1
)

hgb_search.fit(X_train, y_train)

print(f"Best params: {hgb_search.best_params_}")
print(f"Best F1: {hgb_search.best_score_:.4f}")

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params: {'min_samples_leaf': 30, 'max_iter': 150, 'max_depth': 20, 'learning_rate': 0.05}
Best F1: 0.9134


In [9]:
predict_and_report_val(hgb_search.best_estimator_, "Gradient Boosting")

======= Gradient Boosting — Validation ========
  PR-AUC:    0.9147
  F1:        0.7512
  Precision: 0.6465
  Recall:    0.8965

              precision    recall  f1-score   support

 No Purchase       0.98      0.90      0.94    191885
    Purchase       0.65      0.90      0.75     38377

    accuracy                           0.90    230262
   macro avg       0.81      0.90      0.84    230262
weighted avg       0.92      0.90      0.91    230262



In [10]:
predict_and_report_test(hgb_search.best_estimator_, "Gradient Boosting")

======= Gradient Boosting — Test ========
  PR-AUC:    0.9244
  F1:        0.7770
  Precision: 0.6876
  Recall:    0.8932

              precision    recall  f1-score   support

 No Purchase       0.98      0.92      0.95    184365
    Purchase       0.69      0.89      0.78     36873

    accuracy                           0.91    221238
   macro avg       0.83      0.91      0.86    221238
weighted avg       0.93      0.91      0.92    221238



## Subgroup Analysis

In [11]:
# Load test IDs and match with cluster assignments
with open(processed_data_path / 'test_ids.pkl', 'rb') as f:
    test_ids = pickle.load(f)

assert len(test_ids) == len(X_test)
assert len(test_ids) == len(y_test)


clusters = pd.read_csv(data_path / 'processed' / 'customer_clusters_kmeans.csv', 
                       low_memory=False)[['customer_id', 'segment_name']]

test_ids = test_ids.merge(clusters, on='customer_id', how='left')
test_ids['segment_name'] = test_ids['segment_name'].fillna('Unknown')

print(f"Test set size: {len(test_ids):,}")
print(f"\nSegment distribution:")
print(test_ids['segment_name'].value_counts())
print(f"\nUnmatched customers: {(test_ids['segment_name'] == 'Unknown').sum():,}")

Test set size: 221,238

Segment distribution:
segment_name
High-Value Loyalists           117282
Steady Omnichannel Shoppers     43734
At-Risk / One-Time Buyers       30120
Occasional Customers            30102
Name: count, dtype: int64

Unmatched customers: 0


In [12]:
segment_labels = test_ids['segment_name'].values
segments = ['High-Value Loyalists', 'Steady Omnichannel Shoppers', 
            'At-Risk / One-Time Buyers', 'Occasional Customers']

for model, name in [(rf_search.best_estimator_, 'Random Forest'), 
                    (hgb_search.best_estimator_, 'Gradient Boosting')]:
    y_pred = model.predict(X_test)
    
    print(f"\n====== {name} — Subgroup Analysis ======")
    print(f"{'Segment':<35} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    print("-" * 75)
    for seg in segments:
        mask = segment_labels == seg
        p = precision_score(y_test[mask], y_pred[mask], zero_division=0)
        r = recall_score(y_test[mask], y_pred[mask], zero_division=0)
        f = f1_score(y_test[mask], y_pred[mask], zero_division=0)
        n = mask.sum()
        print(f"{seg:<35} {p:>10.3f} {r:>10.3f} {f:>10.3f} {n:>10,}")


====== Random Forest — Subgroup Analysis ======
Segment                              Precision     Recall         F1    Support
---------------------------------------------------------------------------
High-Value Loyalists                     0.738      0.908      0.814    117,282
Steady Omnichannel Shoppers              0.438      0.660      0.526     43,734
At-Risk / One-Time Buyers                0.966      0.992      0.979     30,120
Occasional Customers                     0.949      0.987      0.967     30,102

====== Gradient Boosting — Subgroup Analysis ======
Segment                              Precision     Recall         F1    Support
---------------------------------------------------------------------------
High-Value Loyalists                     0.729      0.911      0.810    117,282
Steady Omnichannel Shoppers              0.397      0.713      0.510     43,734
At-Risk / One-Time Buyers                0.965      0.991      0.978     30,120
Occasional Customers      

In [13]:
thresholds = [0.25, 0.5, 0.75]
target_segments = ['High-Value Loyalists', 'Steady Omnichannel Shoppers']

for model, name in [(rf_search.best_estimator_, 'Random Forest'), 
                    (hgb_search.best_estimator_, 'Gradient Boosting')]:
    y_proba = model.predict_proba(X_test)[:, 1]
    
    print(f"\n====== {name} — Threshold Sensitivity ======")
    print(f"{'Segment':<35} {'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    print("-" * 80)
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        for seg in target_segments:
            mask = segment_labels == seg
            p = precision_score(y_test[mask], y_pred[mask], zero_division=0)
            r = recall_score(y_test[mask], y_pred[mask], zero_division=0)
            f = f1_score(y_test[mask], y_pred[mask], zero_division=0)
            print(f"{seg:<35} {t:>10.2f} {p:>10.3f} {r:>10.3f} {f:>10.3f}")
        print()


====== Random Forest — Threshold Sensitivity ======
Segment                              Threshold  Precision     Recall         F1
--------------------------------------------------------------------------------
High-Value Loyalists                      0.25      0.526      0.970      0.682
Steady Omnichannel Shoppers               0.25      0.275      0.867      0.417

High-Value Loyalists                      0.50      0.738      0.908      0.814
Steady Omnichannel Shoppers               0.50      0.438      0.660      0.526

High-Value Loyalists                      0.75      0.915      0.820      0.865
Steady Omnichannel Shoppers               0.75      0.689      0.336      0.452


====== Gradient Boosting — Threshold Sensitivity ======
Segment                              Threshold  Precision     Recall         F1
--------------------------------------------------------------------------------
High-Value Loyalists                      0.25      0.545      0.967      0.697
Stead

In [ ]:
## Subgroup Analysis — Decision Forest Models

### Results at Default Threshold (0.5)

| Segment | RF F1 | GB F1 | NN F1 |
|---|---|---|---|
| High-Value Loyalists | 0.814 | 0.810 | 0.724 |
| Steady Omnichannel Shoppers | 0.526 | 0.510 | 0.536 |
| At-Risk / One-Time Buyers | 0.979 | 0.978 | 0.944 |
| Occasional Customers | 0.967 | 0.967 | 0.938 |

The pattern mirrors the neural network — At-Risk and Occasional segments are very strong, High-Value Loyalists are moderate, and Steady Omnichannel Shoppers are the weakest segment across all three models. This confirms the Omnichannel weakness is a feature set limitation rather than a model-specific issue.

### Threshold Sensitivity

**High-Value Loyalists** respond well to a higher threshold across both models — raising to 0.75 improves F1 to 0.865 (RF) and 0.862 (GB) by gaining precision faster than losing recall. This is consistent with the neural network finding.

**Steady Omnichannel Shoppers** show a notable difference between models — Gradient Boosting at 0.75 improves slightly to F1=0.530, while Random Forest degrades to 0.452. This suggests Gradient Boosting produces more confident and better-calibrated predictions for this segment, giving it an edge over Random Forest specifically for Omnichannel customers.

